# 06. Pre-model Validation & Seeding

Objetivo: Evaluar la rigidez topológica método de Hopkins y extraer restricciones supervisales de propagación (Must-Link y Cannot-Link) junto a coordenadas de inicialización directas.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import scipy.sparse as sp
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def calculate_hopkins_statistic(matrix_array: np.ndarray) -> float:
    """
    Calculates the Hopkins statistic to strictly validate cluster-readiness in the data.
    Approaching 1.0 implies highly clustered data. Approaching 0.5 implies random distribution.
    """
    logger.info("Computing Hopkins statistic.")
    if len(matrix_array) < 10:
        logger.warning("Not enough data to compute Hopkins.")
        return 0.5

    d = matrix_array.shape[1]
    n = len(matrix_array)
    m = int(0.1 * n) # Sampling 10%
    m = max(1, m)

    nbrs = NearestNeighbors(n_neighbors=1).fit(matrix_array)
    
    # Generate random uniformly distributed points
    rand_matrix = np.random.uniform(np.min(matrix_array, axis=0), np.max(matrix_array, axis=0), (m, d))
    
    u_distances, _ = nbrs.kneighbors(rand_matrix, n_neighbors=1)
    u_sum = np.sum(u_distances)
    
    random_idx = np.random.choice(n, m, replace=False)
    w_matrix = matrix_array[random_idx]
    
    w_distances, _ = nbrs.kneighbors(w_matrix, n_neighbors=2)
    # Index 1 matches nearest neighbor excluding self
    w_sum = np.sum(w_distances[:, 1])
    
    hopkins_val = u_sum / (u_sum + w_sum + 1e-5)
    return float(hopkins_val)


## Seeding: Relaciones Topológicas & Centroides

In [ ]:
def extract_cluster_seeds(df: pd.DataFrame, target_col: str, feature_cols: list) -> pd.DataFrame:
    """
    Extracts exact robust centroids from the 209 natively labeled HCPs (ATSEG).
    Used as forced initializations (seeds) in clustering.
    """
    logger.info("Extracting initialization centroids from labeled cohort.")
    df_labeled = df[df[target_col].notnull()]
    
    if df_labeled.empty:
        return pd.DataFrame()
        
    available_features = [f for f in feature_cols if f in df_labeled.columns]
    # Media to be robust to outliers inside the labeled group
    seeds_df = df_labeled.groupby(target_col)[available_features].median().reset_index()
    return seeds_df

def build_relational_matrices(df: pd.DataFrame, target_col: str):
    """
    Constructs strictly optimized sparse matrices enforcing Must-Link and Cannot-Link constraints 
    among the labeled records explicitly.
    """
    logger.info("Building constraints sparse matrices.")
    
    # Finding labeled indices relative to the dataframe index configuration
    df = df.reset_index(drop=True)
    annotated_mask = df[target_col].notnull()
    labeled_idx = df[annotated_mask].index.tolist()
    labels = df.loc[labeled_idx, target_col].values
    
    n = len(df)
    ml_matrix = sp.lil_matrix((n, n), dtype=np.int8)
    cl_matrix = sp.lil_matrix((n, n), dtype=np.int8)
    
    n_annotated = len(labeled_idx)
    
    for i in range(n_annotated):
        for j in range(i + 1, n_annotated):
            global_i = labeled_idx[i]
            global_j = labeled_idx[j]
            
            if labels[i] == labels[j]:
                ml_matrix[global_i, global_j] = 1
                ml_matrix[global_j, global_i] = 1
            else:
                cl_matrix[global_i, global_j] = 1
                cl_matrix[global_j, global_i] = 1
                
    return ml_matrix.tocsr(), cl_matrix.tocsr()
